In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
train_df = pd.read_csv(
"/content/drive/MyDrive/weighted_fusion_results/fusion_train.csv"
)

val_df = pd.read_csv(
"/content/drive/MyDrive/weighted_fusion_results/fusion_val.csv"
)

test_df = pd.read_csv(
"/content/drive/MyDrive/weighted_fusion_results/fusion_test.csv"
)

print(len(train_df), len(val_df), len(test_df))

20970 4494 4494


In [ ]:
features = [
"DR_prob",
"Gl_prob",
"AMD_prob",
"DED_prob",
"is_fundus",
"is_oct",
"is_slitlamp"
]

targets = ["DR","Glaucoma","AMD","DED"]

In [ ]:
X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
y_train = torch.tensor(train_df[targets].values, dtype=torch.float32)

X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
y_val = torch.tensor(val_df[targets].values, dtype=torch.float32)

X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
y_test = torch.tensor(test_df[targets].values, dtype=torch.float32)

In [ ]:
train_loader = DataLoader(
list(zip(X_train,y_train)),
batch_size=256,
shuffle=True
)

val_loader = DataLoader(
list(zip(X_val,y_val)),
batch_size=512
)

In [ ]:
class WeightedFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(7,16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16,8),
            nn.ReLU(),

            nn.Linear(8,4)
        )

    def forward(self,x):
        return self.net(x)

In [ ]:
model = WeightedFusion().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
model.parameters(),
lr=0.001
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    for X,y in train_loader:

        X = X.to(device)
        y = y.to(device)

        logits = model(X)

        loss = criterion(logits,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch",epoch+1,"Loss",loss.item())

Epoch 1 Loss 0.48026803135871887
Epoch 2 Loss 0.35607030987739563
Epoch 3 Loss 0.2472630739212036
Epoch 4 Loss 0.1962004154920578
Epoch 5 Loss 0.13648763298988342
Epoch 6 Loss 0.135600745677948
Epoch 7 Loss 0.12495590001344681
Epoch 8 Loss 0.11892632395029068
Epoch 9 Loss 0.10042285919189453
Epoch 10 Loss 0.07542436569929123
Epoch 11 Loss 0.09701015800237656
Epoch 12 Loss 0.10419351607561111
Epoch 13 Loss 0.06290179491043091
Epoch 14 Loss 0.08469830453395844
Epoch 15 Loss 0.08216136693954468
Epoch 16 Loss 0.0892254114151001
Epoch 17 Loss 0.09355354309082031
Epoch 18 Loss 0.09355807304382324
Epoch 19 Loss 0.08914647251367569
Epoch 20 Loss 0.09275911748409271


In [ ]:
model.eval()

with torch.no_grad():

    val_logits = model(X_val.to(device))

    val_probs = torch.sigmoid(val_logits).cpu().numpy()

for i,d in enumerate(targets):

    auc = roc_auc_score(
        val_df[d],
        val_probs[:,i]
    )

    print(d,"Val AUC:",auc)

DR Val AUC: 0.9823241027671221
Glaucoma Val AUC: 0.9764023951749583
AMD Val AUC: 0.9993489189329499
DED Val AUC: 0.9988731124633762


In [ ]:
model.eval()

with torch.no_grad():

    test_logits = model(X_test.to(device))

    test_probs = torch.sigmoid(test_logits).cpu().numpy()

print("\nFinal Test AUC")

for i,d in enumerate(targets):

    auc = roc_auc_score(
        test_df[d],
        test_probs[:,i]
    )

    print(d,"AUC:",auc)


Final Test AUC
DR AUC: 0.9804410770545542
Glaucoma AUC: 0.9738454911348451
AMD AUC: 0.9991249661794985
DED AUC: 0.9999959763085045


In [ ]:
aucs = []

for i,d in enumerate(targets):

    auc = roc_auc_score(
    test_df[d],
    test_probs[:,i]
    )

    aucs.append(auc)

print("\nMacro AUC:",np.mean(aucs))


Macro AUC: 0.9883518776693505


In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Disease": ["DR","Glaucoma","AMD","DED","Macro"],
    "AUC": [
        roc_auc_score(test_df["DR"], test_probs[:,0]),
        roc_auc_score(test_df["Glaucoma"], test_probs[:,1]),
        roc_auc_score(test_df["AMD"], test_probs[:,2]),
        roc_auc_score(test_df["DED"], test_probs[:,3]),
        np.mean(aucs)
    ]
})

results

,Disease,AUC
0,DR,0.980441
1,Glaucoma,0.973845
2,AMD,0.999125
3,DED,0.999996
4,Macro,0.988352


In [ ]:
save_path = "/content/drive/MyDrive/weighted_fusion_results/weighted_fusion_metrics.csv"

results.to_csv(save_path, index=False)

print("Saved metrics to:", save_path)

Saved metrics to: /content/drive/MyDrive/weighted_fusion_results/weighted_fusion_metrics.csv


In [ ]:
torch.save(
model.state_dict(),
"/content/drive/MyDrive/weighted_fusion_results/weighted_fusion_model.pth"
)